In [1]:
%matplotlib inline
# Cell 1 — parameters
SPREAD_THRESHOLD        = 20    # percentile-point spread below which a variable is 'concentrated'
P_LOW                   = 10    # lower percentile for spread calculation
P_HIGH                  = 90    # upper percentile for spread calculation
ZERO_FRACTION_THRESHOLD = 0.20  # must match step 2; from work order 2026-06-14
ZERO_COVERAGE_THRESHOLD = 0.90  # buffer weight-at-zero fraction → 'outside_active_domain'

In [2]:
# Cell 2 — imports and load Step 2 outputs
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

sys.path.insert(0, '../../../..')
import scripts.shared.db_utils as _dbu

ROOT = Path(_dbu.__file__).parent.parent.parent
OUT  = ROOT / 'output' / 'edop' / 'areas'

raw_df    = pd.read_csv(OUT / 'step2_raw.tsv',    sep='\t', dtype={'hybas_id': 'int64'}, index_col='hybas_id')
matrix_df = pd.read_csv(OUT / 'step2_matrix.tsv', sep='\t', dtype={'hybas_id': 'int64'}, index_col='hybas_id')
meta_df   = pd.read_csv(OUT / 'step2_meta.tsv',   sep='\t', index_col='api_key')

print(f'raw_df    : {raw_df.shape}')
print(f'matrix_df : {matrix_df.shape}')
print(f'meta_df   : {meta_df.shape}')

raw_df    : (9, 55)
matrix_df : (9, 54)
meta_df   : (54, 8)


In [3]:
# Cell 3 — Step 3.1: join weights onto the matrix
#
# Both files use hybas_id as index; coerce to int before joining so float64
# representations (1060041510.0 vs 1060041510) don't cause mismatches.
# Inner join: any basin missing from either side is a mismatch — fail loudly.

weights = raw_df[['weight']].copy()
weights.index = weights.index.astype(int)

matrix = matrix_df.copy()
matrix.index = matrix.index.astype(int)

joined = weights.join(matrix, how='inner')

n_expected = len(weights)
n_joined   = len(joined)
weight_sum = joined['weight'].sum()

print(f'Basins in weights : {n_expected}')
print(f'Basins after join : {n_joined}  ({"OK" if n_joined == n_expected else "MISMATCH"})')
print(f'Weight sum        : {weight_sum:.6f}  ({"OK" if abs(weight_sum - 1.0) < 0.001 else "CHECK"})')

if n_joined != n_expected:
    missing = set(weights.index) - set(matrix.index)
    print(f'Missing hybas_ids: {missing}')

Basins in weights : 9
Basins after join : 9  (OK)
Weight sum        : 1.000000  (OK)


In [ ]:
# Cell 4 — Step 3.2: select block-1 variables
#
# Block 1 handles two typology clusters via the same area-weighted coherence recipe:
#   continental-gradient — smooth spatially-autocorrelated fields (climate, soils, etc.)
#   scale-dependent      — field-like vars whose Moran's I varies across L6/L8
#                          (slope, karst, wetlands, cropland, groundwater, etc.)
#
# network-topology (discharge_annual, discharge_min, discharge_max) → Block 2.
# discharge_max was re-typed from scale-dependent → network-topology in catalog 2026-06-15;
# step2 must be re-run to propagate this to step2_meta.tsv and include it in Block 2.
#
# local-anomaly (river_area) → deferred to Block 5 fallback.

BLOCK1_CLUSTERS = {'continental-gradient', 'scale-dependent'}

block1_vars = meta_df[
    (meta_df['kind'] == 'continuous') &
    (meta_df['typology_cluster'].isin(BLOCK1_CLUSTERS))
].index.tolist()

block1_vars = [v for v in block1_vars if v in joined.columns]

print(f'Block 1 variables ({len(block1_vars)}):')
for v in block1_vars:
    row = meta_df.loc[v]
    print(f'  {v:35s}  cluster={row["typology_cluster"]}  band={row["band"]}  method={row["position_method"]}')

In [6]:
# Cell 5 — Step 3.3: weighted score distributions
#
# For each block-1 variable:
#   1. Pair scores with weights; drop basins where score is null.
#   2. Renormalize surviving weights to sum to 1.
#      (Null = data absence for that basin, not geographic absence;
#       renormalize rather than report a shortfall.)
#   3. Record coverage: how many basins and how much original weight contributed.
#   4. Record weight_at_zero: fraction of buffer weight with score exactly 0.0
#      (used by degenerate-at-floor guard in Cell 7).

distributions = {}   # api_key → {scores, weights, n, coverage_weight, weight_at_zero}

for var in block1_vars:
    col = joined[var].apply(pd.to_numeric, errors='coerce')
    w   = joined['weight']

    mask   = col.notna()
    scores = col[mask].values.astype(float)
    wts    = w[mask].values.astype(float)

    coverage_weight     = wts.sum()
    wts_norm            = wts / coverage_weight
    weight_at_zero_frac = float(wts[scores == 0.0].sum()) if scores.size > 0 else 0.0

    distributions[var] = {
        'scores':          scores,
        'weights':         wts_norm,
        'n':               int(mask.sum()),
        'coverage_weight': round(float(coverage_weight), 4),
        'weight_at_zero':  round(weight_at_zero_frac, 4),
    }

print(f'Distributions assembled for {len(distributions)} variables')
dropped = [(v, d) for v, d in distributions.items() if d['n'] < len(joined)]
if dropped:
    print('Variables with null-dropped basins:')
    for v, d in dropped:
        print(f'  {v}: {d["n"]}/{len(joined)} basins, coverage_weight={d["coverage_weight"]}')
else:
    print('No null-dropped basins in block-1 variables')

waz_hits = [(v, d['weight_at_zero']) for v, d in distributions.items() if d['weight_at_zero'] > 0]
if waz_hits:
    print(f'\nVariables with buffer weight at score=0:')
    for v, waz in sorted(waz_hits, key=lambda x: -x[1]):
        print(f'  {v:35s}  weight_at_zero={waz:.3f}')

Distributions assembled for 19 variables
Variables with null-dropped basins:
  pct_clay: 8/9 basins, coverage_weight=0.8372
  pct_clay_upstream: 8/9 basins, coverage_weight=0.8372
  pct_silt: 8/9 basins, coverage_weight=0.8372
  pct_silt_upstream: 8/9 basins, coverage_weight=0.8372
  pct_sand: 8/9 basins, coverage_weight=0.8372
  pct_sand_upstream: 8/9 basins, coverage_weight=0.8372

Variables with buffer weight at score=0:
  permafrost_extent                    weight_at_zero=1.000
  dist_sink                            weight_at_zero=0.465
  pasture_extent                       weight_at_zero=0.440
  pasture_extent_upstream              weight_at_zero=0.440
  pct_clay_upstream                    weight_at_zero=0.277
  pct_silt_upstream                    weight_at_zero=0.277


In [7]:
# Cell 6 — Step 3.4: coherence statistics
#
# For each variable:
#   weighted_mean = sum(score_i * weight_i)
#   weighted p10, p90 via sorted cumulative weights + linear interpolation
#   spread = p90 - p10  (in percentile points)

def weighted_quantile(scores, weights, q):
    """Weighted quantile via sorted cumulative weights, linear interpolation."""
    sort_idx = np.argsort(scores)
    s = scores[sort_idx]
    w = weights[sort_idx]
    cumw = np.cumsum(w)
    cumw /= cumw[-1]   # ensure sums to 1 after float rounding
    return float(np.interp(q, cumw, s))

stats = {}
for var, d in distributions.items():
    s, w = d['scores'], d['weights']
    wmean = float(np.dot(s, w))
    p10   = weighted_quantile(s, w, P_LOW  / 100)
    p90   = weighted_quantile(s, w, P_HIGH / 100)
    spread = p90 - p10
    stats[var] = {
        'weighted_mean':    round(wmean,  2),
        'p10':              round(p10,    2),
        'p90':              round(p90,    2),
        'spread':           round(spread, 2),
        'n':                d['n'],
        'coverage_weight':  d['coverage_weight'],
    }

stats_df = pd.DataFrame(stats).T.sort_values('spread')
print(f'Coherence statistics — block 1 ({len(stats_df)} variables)')
print(f'Spread threshold T = {SPREAD_THRESHOLD} percentile points')
print()
display(stats_df)

,weighted_mean,p10,p90,spread,n,coverage_weight
permafrost_extent,0.00,0.00,0.00,0.00,9.0,1.0000
temp_yr,97.85,96.29,98.32,2.03,9.0,1.0000
elev_min,63.20,62.02,64.16,2.13,9.0,1.0000
aridity,10.19,6.26,13.40,7.13,9.0,1.0000
temp_yr_upstream,96.45,91.08,99.11,8.03,9.0,1.0000
precip_yr,16.11,9.04,22.86,13.83,9.0,1.0000
pct_sand,80.50,69.32,89.62,20.30,8.0,0.8372
runoff,20.49,10.50,30.99,20.49,9.0,1.0000
pct_clay,26.97,7.48,35.37,27.89,8.0,0.8372
pct_silt,23.72,4.54,34.13,29.59,8.0,0.8372


In [11]:
# Cell 7 — Step 3.5: classify and emit block-1 results
#
# Shared output envelope (all blocks):
#   variable, method, status, representative_score, representative_raw,
#   n_basins, coverage_weight
# Block-1 detail: spread, p10, p90, weight_at_zero
# Block-2 detail: dominant_hybas_id  (null here)

results = []
for var, s in stats.items():
    zf = None
    if var in meta_df.index and 'zero_fraction' in meta_df.columns:
        try:
            zf = float(meta_df.loc[var, 'zero_fraction'])
            if np.isnan(zf):
                zf = None
        except (TypeError, ValueError):
            zf = None

    waz = distributions[var].get('weight_at_zero', 0.0)

    if zf is not None and zf >= ZERO_FRACTION_THRESHOLD and waz >= ZERO_COVERAGE_THRESHOLD:
        status    = 'outside_active_domain'
        rep_score = None
    elif s['spread'] < SPREAD_THRESHOLD:
        status    = 'concentrated'
        rep_score = s['weighted_mean']
    else:
        status    = 'spread'
        rep_score = None

    results.append({
        'variable':             var,
        'method':               'area_weighted',
        'status':               status,
        'representative_score': rep_score,
        'representative_raw':   None,          # native-unit means deferred
        'n_basins':             s['n'],
        'coverage_weight':      s['coverage_weight'],
        # block-1 detail
        'spread':               s['spread'],
        'p10':                  s['p10'],
        'p90':                  s['p90'],
        'weight_at_zero':       waz,
        # block-2 detail (null for block 1)
        'dominant_hybas_id':    None,
    })

results_df = pd.DataFrame(results).set_index('variable').sort_values('spread')

outside      = results_df[results_df['status'] == 'outside_active_domain']
concentrated = results_df[results_df['status'] == 'concentrated']
spread_vars  = results_df[results_df['status'] == 'spread']

print(f'Block 1 results — T = {SPREAD_THRESHOLD}')
print(f'  outside_active_domain : {len(outside)}')
print(f'  concentrated          : {len(concentrated)}')
print(f'  spread                : {len(spread_vars)}')
print()
if len(outside):
    print('=== OUTSIDE ACTIVE DOMAIN ===')
    display(outside[['spread', 'p10', 'p90', 'weight_at_zero', 'n_basins']])
    print()
print('=== CONCENTRATED (representative_score reported) ===')
display(concentrated[['representative_score', 'spread', 'p10', 'p90', 'n_basins', 'coverage_weight']])
print()
print('=== SPREAD (no single value) ===')
display(spread_vars[['spread', 'p10', 'p90', 'n_basins', 'coverage_weight', 'weight_at_zero']])

results_df.to_csv(OUT / 'step3_block1_results.tsv', sep='\t', float_format='%.2f')
print(f'\nSaved step3_block1_results.tsv')


Saved step3_block1_results.tsv


In [12]:
# Cell 8 — Block 2: network-topology (discharge_annual, discharge_min)
#
# Discharge is cumulative — each basin already integrates upstream flow.
# No mean, no area-weighted distribution. Report the dominant river only.
# Dominant = basin with highest annual discharge in the buffer set.
# Both variables read from that one basin; discharge_min > 0 → perennial,
# discharge_min = 0 → seasonal / intermittent.

nt_vars      = meta_df[meta_df['typology_cluster'] == 'network-topology']
dominant_id  = int(raw_df['discharge_yr'].idxmax())
n_total      = len(raw_df)

block2_rows = []
for api_key, row in nt_vars.iterrows():
    score   = round(float(matrix_df.loc[dominant_id, api_key]), 2)
    raw_val = round(float(raw_df.loc[dominant_id, api_key]),    3)
    block2_rows.append({
        'variable':             api_key,
        'method':               'dominant_basin',
        'status':               'dominant',
        'representative_score': score,
        'representative_raw':   raw_val,
        'n_basins':             n_total,
        'coverage_weight':      1.0,
        'spread':               np.nan,
        'p10':                  np.nan,
        'p90':                  np.nan,
        'weight_at_zero':       np.nan,
        'dominant_hybas_id':    dominant_id,
    })

block2_df = pd.DataFrame(block2_rows).set_index('variable')

print(f'Block 2 — network-topology  (dominant basin: hybas_id {dominant_id})')
print(f'  annual discharge (m³/yr) : {raw_df.loc[dominant_id, "discharge_yr"]:.1f}')
print(f'  min    discharge (m³/mn) : {raw_df.loc[dominant_id, "discharge_min"]:.1f}')
print()
display(block2_df[['method', 'status', 'representative_score', 'representative_raw', 'dominant_hybas_id']])

# Align dtypes before concat to avoid FutureWarning about all-NA column inference.
# Block 1 has all-None: dominant_hybas_id (→ Int64) and representative_raw (→ float64).
# Block 2 has all-NaN: spread, p10, p90, weight_at_zero (→ float64, already, but explicit).
results_df['dominant_hybas_id']  = results_df['dominant_hybas_id'].astype('Int64')
block2_df['dominant_hybas_id']   = block2_df['dominant_hybas_id'].astype('Int64')
results_df['representative_raw'] = results_df['representative_raw'].astype('float64')
for col in ['spread', 'p10', 'p90', 'weight_at_zero']:
    block2_df[col] = block2_df[col].astype('float64')

combined_df = pd.concat([results_df, block2_df])
combined_df.to_csv(OUT / 'step3_results.tsv', sep='\t', float_format='%.2f')
print(f'\nSaved step3_results.tsv  ({len(combined_df)} variables total)')


Saved step3_results.tsv  (21 variables total)
